# Lentils × Dinomaly — AdaCLIP-bands training tutorial

Train **Dinomaly** on the 3 spectral bands that **AdaCLIP's frozen concrete selector**
converged to (cube indices **14 / 59 / 57**). Those indices are resolved to wavelengths from
the data (≈ 542 / 902 / 886 nm on the 61-band lentils cubes) and used as a **fixed**
`FixedWavelengthSelector` — the bands are *pre-selected and frozen*, not learned here.

This is the head-to-head-with-AdaCLIP variant: same bands AdaCLIP picked, but reconstructed by
Dinomaly. (A fully *learnable* concrete selector, jointly trained, lives as the standalone
`examples/train_dinomaly_concrete_joint_multifile.py` if you want to compare.)

Dinomaly is unsupervised + reconstruction-based, so it trains on the **308 normal frames** only.

Sibling notebooks — `lentils_rgb_train_tutorial.ipynb`, `lentils_cir_train_tutorial.ipynb`,
and `lentils_inference_tutorial.ipynb` (evaluate + per-class AUROC).

> **Prerequisites**
> - Run from the repo's `notebooks/lentils_anomaly/` folder (imports the sibling `utils.py`
>   and the repo's `examples/plugins.yaml`).
> - Env: cuvis-ai ≥ 0.10, cuvis-ai-core ≥ 0.10, cuvis-ai-schemas ≥ 0.7,
>   cuvis-ai-dataloader ≥ 0.3, anomalib 2.1, a CUDA GPU.
> - `LENTILS_DATA_SOURCE=local` (default) reads the on-server NPZ (correct baked masks); `hf`
>   converts via `ensure_lentils_npz` (needs the cuvis SDK + `cuvis-ai-dataloader[cu3s,coco]`).
>   Verified end-to-end on a subset.
> - Knobs are env-overridable (`LENTILS_MAX_EPOCHS`, `LENTILS_IMAGE_SIZE`, `LENTILS_SMOKE_LIMIT`).

## 1 · Dataset overview

61-channel VNIR (430–910 nm) cubes of lentils on a conveyor; foreign objects (stones, metal
shards, paper, rubber, insects) are the anomalies — 8 COCO categories (0 = normal). Published
at `cubert-gmbh/XMR_Industrial_Foreign_Object_Detection_Lentils`.

**Dinomaly (train-on-normals) split** — `splits_dinomaly.csv`: train 308 (normal) / val 148 /
test 180 / adaclip_train 500 (held out). Dinomaly trains on the 308 normal frames only.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import torch
import pytorch_lightning as pl

sys.path.insert(0, str(Path('.').resolve()))
import utils
from utils import ADACLIP_BAND_INDICES, LENTILS_CATEGORIES, resolve_config, resolve_splits_csv

config = resolve_config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:        ', DEVICE)
print('Dataset source:', config['data_source'])
print('Selector:       AdaCLIP frozen bands (cube indices', ADACLIP_BAND_INDICES, ') -> fixed wavelengths')

# --- Tutorial knobs (env-overridable) ------------------------------------
MAX_EPOCHS  = int(os.environ.get('LENTILS_MAX_EPOCHS', '1'))    # bump to 50 for the full run
IMAGE_SIZE  = int(os.environ.get('LENTILS_IMAGE_SIZE', '448'))  # square side, multiple of 14
SMOKE_LIMIT = int(os.environ.get('LENTILS_SMOKE_LIMIT', '0'))   # 0 = all frames; N = N/split dry-run
OUTPUT_DIR  = Path('outputs/lentils_adaclip_run')              # inference notebook reads this (local mode)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Epochs: {MAX_EPOCHS} | image_size: {IMAGE_SIZE} | smoke_limit: {SMOKE_LIMIT or "(all)"}')

## 2 · Get + prepare the data

A splits CSV of `(split, npz_path, image_id)` that `MultiNpzDataModule` reads. `local` uses
the on-server NPZ directly.

In [ ]:
if config['data_source'] == 'local':
    SPLITS_CSV = resolve_splits_csv()
else:
    SPLITS_CSV = utils.ensure_lentils_npz(OUTPUT_DIR / 'npz', limit=SMOKE_LIMIT)  # HF download -> convert

if SMOKE_LIMIT and config['data_source'] == 'local':
    SPLITS_CSV = utils.subsample_splits_csv(SPLITS_CSV, SMOKE_LIMIT, OUTPUT_DIR / 'splits_smoke.csv')

print('Train splits CSV:', SPLITS_CSV)

## 2.5 · Resolve the AdaCLIP bands + peek

The frozen band **indices** are dataset-independent; their **wavelengths** are read from the
data (the first train NPZ's `wavelengths`). Below we resolve them and preview a labelled frame
in that exact false-color (i.e. what the model actually sees).

In [ ]:
import csv
import matplotlib.pyplot as plt

TARGET_WL = utils.resolve_adaclip_wavelengths(SPLITS_CSV, ADACLIP_BAND_INDICES)
print('AdaCLIP band indices', ADACLIP_BAND_INDICES, '-> wavelengths (R,G,B) nm =',
      tuple(round(w, 1) for w in TARGET_WL))

rows = list(csv.DictReader(open(SPLITS_CSV)))
peek = next((r for r in rows if r.get('split') in ('test', 'val')), rows[0])
fr = utils.load_lentils_frame(peek['npz_path'])
view = utils.false_color(fr['cube'], fr['wavelengths'], TARGET_WL)
present = sorted({int(c) for c in np.unique(fr['class_mask']) if int(c) != 0})
labels = [LENTILS_CATEGORIES.get(c, str(c)) for c in present]
print('frame:', Path(peek['npz_path']).name, '| classes present:', labels or '(normal)')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(view); ax[0].set_title('AdaCLIP-bands false color'); ax[0].axis('off')
ax[1].imshow(view)
if fr['mask'].any():
    ax[1].contour(fr['mask'] > 0, levels=[0.5], colors='red', linewidths=1.2)
ax[1].set_title('scene + GT contour'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 3 · Build the 3-channel pipeline

```
LentilsAnomalyDataNode -> MinMaxNormalizer -> FixedWavelengthSelector(AdaCLIP bands)
     -> DinomalyDetector(input_channels=3) -> {QuantileBinaryDecider, AnomalyDetectionMetrics,
        AnomalyAUROCMetrics} -> TensorBoardMonitorNode
```

Identical to the RGB/CIR pipelines except the selector's three target wavelengths are the
AdaCLIP-frozen bands resolved above. The selector is **fixed** (no learnable parameters).

In [ ]:
from cuvis_ai.node.data import LentilsAnomalyDataNode
from cuvis_ai.node.normalization import MinMaxNormalizer
from cuvis_ai.node.channel_selector import FixedWavelengthSelector
from cuvis_ai.node.metrics import AnomalyDetectionMetrics
from cuvis_ai.node.monitor import TensorBoardMonitorNode
from cuvis_ai.deciders.binary_decider import QuantileBinaryDecider
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_dinomaly.node.dinomaly_detector import DinomalyDetector
from cuvis_ai_dinomaly.node.dinomaly_train_loss_bridge import DinomalyTrainLossBridge
from cuvis_ai_dinomaly.node.auroc_metrics import AnomalyAUROCMetrics

pl.seed_everything(42, workers=True)

datamodule = MultiNpzDataModule(splits_csv=str(SPLITS_CSV), batch_size=1, num_workers=0)
datamodule.setup(stage='fit')

pipeline = CuvisPipeline('dinomaly_lentils_adaclip')
data_node  = LentilsAnomalyDataNode(normal_class_ids=[0])
normalizer = MinMaxNormalizer(eps=1e-6, use_running_stats=True, max_initialization_frames=20)
selector   = FixedWavelengthSelector(target_wavelengths=TARGET_WL, name='adaclip_selector')
selector._requires_initial_fit_override = False
dinomaly   = DinomalyDetector(
    encoder_name='dinov2reg_vit_base_14', bottleneck_dropout=0.2, decoder_depth=8,
    image_size=IMAGE_SIZE, crop_size=IMAGE_SIZE, use_center_crop=False,
    input_channels=3, name='dinomaly_detector')
loss_bridge  = DinomalyTrainLossBridge(weight=1.0, name='dinomaly_train_loss')
decider      = QuantileBinaryDecider(quantile=0.995, name='decider')
metrics_node = AnomalyDetectionMetrics(name='metrics_anomaly')
auroc_node   = AnomalyAUROCMetrics(name='metrics_auroc')
tb           = TensorBoardMonitorNode(output_dir=str(OUTPUT_DIR / 'tensorboard'), run_name=pipeline.name)

pipeline.connect(
    (data_node.outputs.cube,            normalizer.data),
    (normalizer.normalized,             selector.cube),
    (data_node.outputs.wavelengths,     selector.wavelengths),
    (selector.rgb_image,                dinomaly.rgb_image),
    (dinomaly.outputs.training_loss,    loss_bridge.raw_loss),
    (dinomaly.outputs.scores,           decider.logits),
    (dinomaly.outputs.scores,           metrics_node.logits),
    (decider.decisions,                 metrics_node.decisions),
    (data_node.outputs.mask,            metrics_node.targets),
    (metrics_node.metrics,              tb.metrics),
    (dinomaly.outputs.scores,           auroc_node.scores),
    (data_node.outputs.mask,            auroc_node.targets),
    (dinomaly.outputs.anomaly_score,    auroc_node.anomaly_score),
)
print('Pipeline:', [n.name for n in pipeline.nodes])

## 4 · Train

Two phases: `StatisticalTrainer` initialises the MinMax normaliser's running bounds, then
`GradientTrainer` trains the Dinomaly bottleneck + decoder (the DINOv2 encoder stays frozen).
Optimizer + LR match `configs/trainrun/dinomaly_multifile_rgb_frozen_adaclip_bands.yaml`
(AdamW, lr 2e-3).

In [ ]:
from cuvis_ai_core.training import GradientTrainer, StatisticalTrainer
from cuvis_ai_core.training.config import create_callbacks_from_config
from cuvis_ai_schemas.training import (
    CallbacksConfig, ModelCheckpointConfig, OptimizerConfig, TrainerConfig,
)

trainer_cfg = TrainerConfig(
    max_epochs=MAX_EPOCHS, accelerator='auto', devices=1,
    default_root_dir=str(OUTPUT_DIR), precision='32-true',
    enable_progress_bar=True, enable_checkpointing=True, log_every_n_steps=10,
    check_val_every_n_epoch=1, gradient_clip_val=0.1,
    callbacks=CallbacksConfig(checkpoint=ModelCheckpointConfig(
        dirpath=str(OUTPUT_DIR / 'checkpoints'), monitor='metrics_anomaly/iou',
        mode='max', save_top_k=1, save_last=True, filename='{epoch:02d}')),
)
optimizer_cfg = OptimizerConfig(name='adamw', lr=2e-3, weight_decay=1e-4, betas=[0.9, 0.999])

if normalizer.requires_initial_fit:
    print('Phase 1: statistical initialization (MinMaxNormalizer)...')
    StatisticalTrainer(pipeline=pipeline, datamodule=datamodule).fit()

pipeline.unfreeze_nodes_by_name(['dinomaly_detector'])
pipeline.to(DEVICE)
n_train = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in pipeline.parameters())
print(f'Trainable: {n_train:,} / {n_total:,} ({100*n_train/max(n_total,1):.2f}%)')

callbacks = list(create_callbacks_from_config(trainer_cfg.callbacks)) if trainer_cfg.callbacks else []
print(f'Phase 2: gradient training for {MAX_EPOCHS} epoch(s)...')
grad_trainer = GradientTrainer(
    pipeline=pipeline, datamodule=datamodule,
    loss_nodes=[loss_bridge], metric_nodes=[metrics_node, auroc_node],
    trainer_config=trainer_cfg, optimizer_config=optimizer_cfg,
    monitors=[tb], callbacks=callbacks,
)
grad_trainer.fit()
print('Training done.')

## 5 · Save the trained pipeline

Writes `<OUTPUT_DIR>/trained_models/dinomaly_lentils_adaclip.yaml` + `.pt`. Point the
inference notebook at it with `LENTILS_PIPELINE_DIR`.

In [ ]:
from cuvis_ai_schemas.pipeline import PipelineMetadata

results_dir = OUTPUT_DIR / 'trained_models'
results_dir.mkdir(parents=True, exist_ok=True)
pipeline_path = results_dir / f'{pipeline.name}.yaml'
pipeline.save_to_file(
    str(pipeline_path),
    metadata=PipelineMetadata(
        name=pipeline.name,
        description=f'Dinomaly on lentils VNIR NPZ, fixed AdaCLIP-frozen bands {ADACLIP_BAND_INDICES} '
                    f'-> {tuple(round(w,1) for w in TARGET_WL)} nm, 3-channel.',
        tags=['dinomaly', 'anomalib', 'lentils', 'adaclip', 'hyperspectral'],
        author='cuvis.ai'),
)
print('Saved:', pipeline_path)
pt = pipeline_path.with_suffix('.pt')
print('Weights:', pt, f'({pt.stat().st_size/1e6:.0f} MB)' if pt.is_file() else '(missing)')

## 6 · Next

Evaluate with **`lentils_inference_tutorial.ipynb`** and
`LENTILS_PIPELINE_DIR=notebooks/lentils_anomaly/outputs/lentils_adaclip_run/trained_models`
for the per-class AUROC breakdown on the 180 test. For the full run: `LENTILS_MAX_EPOCHS=50`
here, or `uv run python examples/train_dinomaly_rgb_frozen_adaclip_bands_multifile.py data.splits_csv=<your CSV>`.